In [ ]:
# ============================
# CELL 1: Install Dependencies
# ============================
!pip install -q segmentation-models-pytorch albumentations torchmetrics tqdm


In [ ]:
# ============================
# CELL 2: GPU Setup & Seed
# ============================
import torch, random, numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================
# CELL 3: Imports
# ============================
import os, glob, cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp


In [ ]:
# ============================
# CELL 4: Dataset Loader
# ============================

DATASET_PATH = r"C:\RESERACH\SAR custom dataset"

IMAGE_DIR = os.path.join(DATASET_PATH, "images")
MASK_DIR  = os.path.join(DATASET_PATH, "masks")

if os.path.exists(os.path.join(IMAGE_DIR, "train")):
    train_imgs  = sorted(glob.glob(os.path.join(IMAGE_DIR, "train", "*.*")))
    train_masks = sorted(glob.glob(os.path.join(MASK_DIR, "train", "*.*")))
    val_imgs    = sorted(glob.glob(os.path.join(IMAGE_DIR, "val", "*.*")))
    val_masks   = sorted(glob.glob(os.path.join(MASK_DIR, "val", "*.*")))
    test_imgs, test_masks = val_imgs, val_masks
else:
    images = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.*")))
    masks  = sorted(glob.glob(os.path.join(MASK_DIR, "*.*")))
    train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
        images, masks, test_size=0.3, random_state=SEED)
    val_imgs, test_imgs, val_masks, test_masks = train_test_split(
        temp_imgs, temp_masks, test_size=0.5, random_state=SEED)

print("Train:", len(train_imgs), "Val:", len(val_imgs))


In [ ]:
# ============================
# CELL 5: SAR Augmentations
# ============================

IMAGE_SIZE = 256

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=30, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.RandomResizedCrop(size=(IMAGE_SIZE, IMAGE_SIZE), scale=(0.8,1.0), p=0.5),
    A.RandomBrightnessContrast(0.15,0.15,p=0.5),
    A.GaussNoise(var_limit=(5.0,20.0),p=0.3),
    A.CLAHE(clip_limit=2.0,p=0.3),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE,IMAGE_SIZE),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])

In [ ]:
# ============================
# CELL 6: Dataset Class
# ============================

class SARDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = cv2.imread(self.images[idx], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype("float32")

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"].unsqueeze(0)

        return img, mask


In [ ]:
# ============================
# CELL 7: DataLoaders
# ============================

train_ds = SARDataset(train_imgs, train_masks, train_transform)
val_ds   = SARDataset(val_imgs, val_masks, val_transform)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0)


In [ ]:
# ============================
# CELL 8: Hybrid Dual-Branch Model (Stable)
# ============================

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

class HybridFusionModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Branch 1: U-Net
        self.unet = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=1,
            classes=1,
            activation=None
        )

        # Branch 2: DeepLabV3+
        self.deeplab = smp.DeepLabV3Plus(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=1,
            classes=1,
            activation=None
        )

        # Fusion layer
        self.fusion = nn.Conv2d(2, 1, kernel_size=1)

    def forward(self, x):

        out_unet = self.unet(x)
        out_deeplab = self.deeplab(x)

        # Concatenate along channel dimension
        combined = torch.cat([out_unet, out_deeplab], dim=1)

        fused_output = self.fusion(combined)

        return fused_output


model = HybridFusionModel().to(DEVICE)

print("Hybrid U-Net + DeepLabV3 Fusion Model Loaded ✅")

In [ ]:
# ============================
# CELL 9: Dice + Focal + BCE Loss
# ============================

dice_loss  = smp.losses.DiceLoss(mode="binary")
focal_loss = smp.losses.FocalLoss(mode="binary")
bce_loss   = nn.BCEWithLogitsLoss()

def loss_fn(pred, target):
    return 0.5*dice_loss(pred,target) +            0.3*focal_loss(pred,target) +            0.2*bce_loss(pred,target)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

In [ ]:
# ============================
# CELL 10: Training Loop 
# ============================



from sklearn.metrics import accuracy_score, precision_score, recall_score
import numpy as np

scaler = torch.cuda.amp.GradScaler()
EPOCHS = 20

# ============================
# Metric Storage Lists
# ============================

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

train_ious = []
val_ious = []

train_dices = []
val_dices = []

train_precisions = []
val_precisions = []

train_recalls = []
val_recalls = []

lr_history = []

# ============================
# Helper Metric Functions
# ============================

def compute_iou(y_true, y_pred):
    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return intersection / (union + 1e-8)

def compute_dice(y_true, y_pred):
    intersection = (y_true * y_pred).sum()
    return (2. * intersection) / (y_true.sum() + y_pred.sum() + 1e-8)

# ============================
# Training Loop
# ============================

for epoch in range(EPOCHS):

    # -------------------
    # TRAINING
    # -------------------
    model.train()
    total_loss = 0

    train_preds = []
    train_targets = []

    for imgs, masks in tqdm(train_loader):

        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = loss_fn(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).float()

        train_preds.extend(preds.detach().cpu().numpy().ravel())
        train_targets.extend(masks.detach().cpu().numpy().ravel())

    scheduler.step()

    # Store LR
    if hasattr(optimizer, "param_groups"):
        lr_history.append(optimizer.param_groups[0]['lr'])

    epoch_train_loss = total_loss / len(train_loader)

    train_preds_np = np.array(train_preds)
    train_targets_np = np.array(train_targets)

    epoch_train_acc = accuracy_score(train_targets_np, train_preds_np)
    epoch_train_prec = precision_score(train_targets_np, train_preds_np, zero_division=0)
    epoch_train_rec = recall_score(train_targets_np, train_preds_np, zero_division=0)
    epoch_train_iou = compute_iou(train_targets_np, train_preds_np)
    epoch_train_dice = compute_dice(train_targets_np, train_preds_np)

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)
    train_precisions.append(epoch_train_prec)
    train_recalls.append(epoch_train_rec)
    train_ious.append(epoch_train_iou)
    train_dices.append(epoch_train_dice)

    # -------------------
    # VALIDATION
    # -------------------
    model.eval()
    val_total_loss = 0

    val_preds = []
    val_targets = []

    with torch.no_grad():
        for imgs, masks in val_loader:

            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

            outputs = model(imgs)
            loss = loss_fn(outputs, masks)

            val_total_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).float()

            val_preds.extend(preds.cpu().numpy().ravel())
            val_targets.extend(masks.cpu().numpy().ravel())

    epoch_val_loss = val_total_loss / len(val_loader)

    val_preds_np = np.array(val_preds)
    val_targets_np = np.array(val_targets)

    epoch_val_acc = accuracy_score(val_targets_np, val_preds_np)
    epoch_val_prec = precision_score(val_targets_np, val_preds_np, zero_division=0)
    epoch_val_rec = recall_score(val_targets_np, val_preds_np, zero_division=0)
    epoch_val_iou = compute_iou(val_targets_np, val_preds_np)
    epoch_val_dice = compute_dice(val_targets_np, val_preds_np)

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)
    val_precisions.append(epoch_val_prec)
    val_recalls.append(epoch_val_rec)
    val_ious.append(epoch_val_iou)
    val_dices.append(epoch_val_dice)

    # -------------------
    # Epoch Summary Print
    # -------------------

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val IoU: {epoch_val_iou:.4f} | "
        f"Val Dice: {epoch_val_dice:.4f}"
    )

In [ ]:
# ============================
# CELL 11: Validation Metrics (%)
# ============================

model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        
        all_preds.extend(preds.cpu().numpy().ravel())
        all_targets.extend(masks.numpy().ravel())

accuracy  = accuracy_score(all_targets, all_preds)
precision = precision_score(all_targets, all_preds, zero_division=0)
recall    = recall_score(all_targets, all_preds, zero_division=0)
f1        = f1_score(all_targets, all_preds, zero_division=0)

intersection = np.logical_and(all_preds, all_targets).sum()
union = np.logical_or(all_preds, all_targets).sum()
iou = intersection / (union + 1e-7)
dice = (2*intersection)/(np.sum(all_preds)+np.sum(all_targets)+1e-7)

print("\n====== VALIDATION METRICS ======")
print(f"Accuracy : {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall   : {recall*100:.2f}%")
print(f"F1       : {f1*100:.2f}%")
print(f"IoU      : {iou*100:.2f}%")
print(f"Dice     : {dice*100:.2f}%")
print("=================================")


In [ ]:
# ============================
# Combined Rows 
# ============================

model.eval()
threshold = 0.5

# From first combined block
indices_block1 = [5, 10, 11]

# From 5-row block
start_index = 5
num_rows_block2 = 5

second_row_block2 = start_index + 1        # 6
last_row_block2   = start_index + num_rows_block2 - 1  # 9

# Final combined indices
indices = indices_block1 + [last_row_block2, second_row_block2]

num_rows = len(indices)

plt.figure(figsize=(12, 15))

with torch.no_grad():
    for i, idx in enumerate(indices):

        img, mask = val_ds[idx]
        img_gpu = img.unsqueeze(0).to(DEVICE)

        output = model(img_gpu)
        prob = torch.sigmoid(output)[0]
        pred = (prob > threshold).float()

        sar_img = img[0].cpu().numpy()
        gt_mask = mask[0].cpu().numpy()
        pred_mask = pred[0].cpu().numpy()

        # SAR
        plt.subplot(num_rows, 3, i*3 + 1)
        plt.imshow(sar_img, cmap='gray')
        plt.title("SAR Image")
        plt.axis("off")

        # GT
        plt.subplot(num_rows, 3, i*3 + 2)
        plt.imshow(gt_mask, cmap='gray')
        plt.title("Ground Truth")
        plt.axis("off")

        # Prediction
        plt.subplot(num_rows, 3, i*3 + 3)
        plt.imshow(pred_mask, cmap='gray')
        plt.title("Prediction")
        plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================
# CELL: Loss Curve
# ============================

import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss Curve")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ============================
# CELL: Confusion Matrix
# ============================

from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = (torch.sigmoid(outputs) > 0.5).float()

        all_preds.extend(preds.cpu().numpy().ravel())
        all_targets.extend(masks.numpy().ravel())

cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Background", "Oil"],
            yticklabels=["Background", "Oil"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Pixel-wise Confusion Matrix")
plt.show()

In [ ]:
# ===============================
# CELL 16: Training & Validation Accuracy Curve 
# ===============================

import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

if 'train_accuracies' not in globals() or 'val_accuracies' not in globals():
    print("Run training cell first.")
else:
    epochs = np.arange(len(train_accuracies))

    plt.figure(figsize=(9,6))

    plt.plot(epochs, train_accuracies,
             linewidth=2.8,
             label='Train Accuracy')

    plt.plot(epochs, val_accuracies,
             linewidth=2.8,
             label='Validation Accuracy')

    # 🔥 Dynamic zoom
    all_values = train_accuracies + val_accuracies
    ymin = min(all_values) - 0.01
    ymax = max(all_values) + 0.01

    plt.ylim(ymin, ymax)

    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title('Training & Validation Accuracy Curve', fontsize=14)

    plt.legend(loc='lower right', fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

In [ ]:
# ===============================
# CELL 12: Class Pixel Distribution (Oil vs Background)
# ===============================

model.eval()

oil_pixels = 0
background_pixels = 0

with torch.no_grad():
    for _, masks in train_loader:
        masks = masks.to(DEVICE)
        oil_pixels += masks.sum().item()
        background_pixels += masks.numel() - masks.sum().item()

total_pixels = oil_pixels + background_pixels
oil_percent = (oil_pixels / total_pixels) * 100
bg_percent = (background_pixels / total_pixels) * 100

plt.figure(figsize=(6,6))
bars = plt.bar(['Oil Spill', 'Background'],
               [oil_percent, bg_percent])

plt.ylabel('Percentage (%)')
plt.title('Class Pixel Distribution (Training Set)')
plt.ylim(0,100)

for i, v in enumerate([oil_percent, bg_percent]):
    plt.text(i, v + 1, f"{v:.2f}%", ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# CELL 13: SAR Backscatter Intensity Distribution
# ===============================

all_pixels = []

with torch.no_grad():
    for images, _ in train_loader:
        images = images.to(DEVICE)
        all_pixels.append(images.view(-1).cpu().numpy())

all_pixels = np.concatenate(all_pixels)

plt.figure(figsize=(8,5))
plt.hist(all_pixels, bins=100)
plt.xlabel('Pixel Intensity')
plt.ylabel('Frequency')
plt.title('SAR Backscatter Intensity Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# CELL 15: Segmentation Metrics Over Epochs
# ===============================

plt.figure(figsize=(8,6))

if 'train_ious' in globals():
    plt.plot(train_ious, label='IoU')
if 'train_dices' in globals():
    plt.plot(train_dices, label='Dice')
if 'train_precisions' in globals():
    plt.plot(train_precisions, label='Precision')
if 'train_recalls' in globals():
    plt.plot(train_recalls, label='Recall')

plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Segmentation Metrics Over Epochs')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# CELL 17: Boundary Quality Visualization (Random Different Image)
# ===============================



import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

model.eval()

# 🔥 Pick a random image from validation dataset
image_index = random.randint(0, len(val_loader.dataset) - 1)
print(f"Showing validation image index: {image_index}")

img, mask = val_loader.dataset[image_index]

img = img.unsqueeze(0).to(DEVICE)
mask = mask.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    output = model(img)
    pred = (torch.sigmoid(output) > 0.5).float()

# Convert to numpy safely
img_np = img[0].cpu().squeeze().numpy()
gt_np = mask[0].cpu().squeeze().numpy()
pred_np = pred[0].cpu().squeeze().numpy()

# Normalize SAR image properly for visualization
img_vis = cv2.normalize(img_np, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Extract boundaries
edges = cv2.Canny((pred_np * 255).astype(np.uint8), 50, 150)

overlay = cv2.cvtColor(img_vis, cv2.COLOR_GRAY2BGR)
overlay[edges > 0] = [255, 0, 0]  # Red boundary

plt.figure(figsize=(12,8))

plt.subplot(2,2,1)
plt.imshow(img_np, cmap='gray')
plt.title('Original SAR')
plt.axis('off')

plt.subplot(2,2,2)
plt.imshow(gt_np, cmap='gray')
plt.title('Ground Truth Mask')
plt.axis('off')

plt.subplot(2,2,3)
plt.imshow(pred_np, cmap='gray')
plt.title('Predicted Binary Mask')
plt.axis('off')

plt.subplot(2,2,4)
plt.imshow(overlay)
plt.title('Boundary Overlay')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# CELL 18: ROC Curve (Pixel-Level)
# ===============================

from sklearn.metrics import roc_curve, auc

model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        outputs = torch.sigmoid(model(images))
        all_probs.append(outputs.view(-1).cpu().numpy())
        all_labels.append(masks.view(-1).cpu().numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# CELL XX: Grad-CAM 
# ===============================

# Every time you RUN this cell, it will automatically
# select a RANDOM validation image.

import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

model.eval()

# 🔥 Pick random validation image
image_index = random.randint(0, len(val_loader.dataset) - 1)
print(f"Grad-CAM for validation image index: {image_index}")

img, mask = val_loader.dataset[image_index]
img = img.unsqueeze(0).to(DEVICE)

# -----------------------
# Automatically get last Conv2D layer
# -----------------------
target_layer = None
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        target_layer = module

activations = []
gradients = []

def forward_hook(module, input, output):
    activations.append(output)

def backward_hook(module, grad_input, grad_output):
    gradients.append(grad_output[0])

handle_f = target_layer.register_forward_hook(forward_hook)
handle_b = target_layer.register_backward_hook(backward_hook)

# -----------------------
# Forward + Backward pass
# -----------------------
output = model(img)

# Oil class activation
score = output[:, 0, :, :].mean()

model.zero_grad()
score.backward()

# -----------------------
# Generate Grad-CAM
# -----------------------
grad = gradients[0]
act = activations[0]

weights = grad.mean(dim=(2,3), keepdim=True)
cam = (weights * act).sum(dim=1, keepdim=True)
cam = F.relu(cam)

cam = F.interpolate(cam, size=img.shape[2:], mode='bilinear', align_corners=False)

cam = cam.squeeze().detach().cpu().numpy()
cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

# -----------------------
# Visualization
# -----------------------
img_np = img.squeeze().cpu().numpy()
img_vis = cv2.normalize(img_np, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)

overlay = cv2.addWeighted(
    cv2.cvtColor(img_vis, cv2.COLOR_GRAY2BGR),
    0.6,
    heatmap,
    0.4,
    0
)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img_np, cmap='gray')
plt.title("Original SAR")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB))
plt.title("Grad-CAM Heatmap")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title("Overlay Visualization")
plt.axis("off")

plt.suptitle("Grad-CAM Visualization for Model Attention", fontsize=13)
plt.tight_layout()
plt.show()

handle_f.remove()
handle_b.remove()